In [ ]:
KE_SRC = 'import os, glob, time, gc\nos.environ.setdefault(\'HF_HUB_OFFLINE\', \'1\')\nos.environ.setdefault(\'TRANSFORMERS_OFFLINE\', \'1\')\nos.environ.setdefault(\'HF_HUB_DISABLE_TELEMETRY\', \'1\')\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\nimport timm\ntorch.backends.cudnn.benchmark = True\ntorch.backends.cuda.matmul.allow_tf32 = True\nIMG = 336\nCROP_MM = 140.0\nSLOTS = [(\'Sagittal\', 1, 18), (\'Sagittal\', 0, 14), (\'Coronal\', 1, 12), (\'Coronal\', 0, 8), (\'Axial\', -1, 12)]\nMAXS = sum((s[2] for s in SLOTS))\nK_EVAL = 42\nNORM = \'imagenet\'\nLAB = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\', \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)\n_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)\nARMS = [{\'file\': \'raptor_ft_coatnet_v5_full_swa.pt\', \'arch\': \'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k\', \'res\': 384, \'w\': 1.0}]\n\ndef build_backbone(arch, pretrained=False):\n    hybrid = arch.startswith((\'maxvit\', \'maxxvit\', \'coatnet\', \'coat_\', \'convnext\'))\n    is_vit = not hybrid and any((k in arch for k in (\'vit\', \'deit\', \'dinov2\', \'eva\', \'beit\')))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)\n    if is_vit:\n        kw.update(global_pool=\'token\', dynamic_img_size=True)\n    else:\n        kw.update(global_pool=\'avg\')\n    return timm.create_model(arch, **kw)\n\nclass RaptorClassifier(nn.Module):\n\n    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        f = self.backbone(x.flatten(0, 1))\n        return f.view(B, K, -1)\n\n    def head(self, feats):\n        h = self.norm(feats)\n        a = self.att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum(\'bkn,bkf->bnf\', a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def forward(self, x):\n        return self.head(self.encode(x))\n\ndef load_model(pt_path, arch_default, res_default, device, ngpu=1):\n    ck = torch.load(pt_path, map_location=\'cpu\', weights_only=False)\n    arch = ck.get(\'arch\', arch_default)\n    ck_res = int(ck.get(\'res\', res_default))\n    bb = build_backbone(arch, pretrained=False)\n    model = RaptorClassifier(bb, F_dim=bb.num_features)\n    model.load_state_dict(ck[\'model\'], strict=True)\n    model.eval().to(device)\n    del ck\n    gc.collect()\n    return (model, ck_res)\n\ndef _eval_centers(mask, D, k):\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3:\n        valid = np.arange(min(3, D))\n    lo, hi = (int(valid.min()), int(valid.max()))\n    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]\n    if not cs:\n        cs = [max(1, min((lo + hi) // 2, D - 2))]\n    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)\n    return [cs[i] for i in idx]\n\ndef eval_windows(vol, mask, k, res, norm=NORM):\n    D = vol.shape[0]\n    cs = _eval_centers(mask, D, k)\n    wins = np.empty((len(cs), 3, res, res), np.float32)\n    for j, c in enumerate(cs):\n        c = max(1, min(c, D - 2))\n        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0\n        t = torch.from_numpy(tri)\n        if t.shape[-1] != res:\n            t = F.interpolate(t[None], size=(res, res), mode=\'bilinear\', align_corners=False)[0]\n        wins[j] = t.numpy()\n    x = torch.from_numpy(wins)\n    if norm == \'imagenet\':\n        x = (x - _MEAN) / _STD\n    return x\n\n@torch.no_grad()\ndef infer_probs(model, xwins, device):\n    x = xwins.unsqueeze(0).to(device)\n    use_cuda = device != \'cpu\' and str(device).startswith(\'cuda\')\n    if use_cuda:\n        try:\n            with torch.autocast(\'cuda\', dtype=torch.float16):\n                o = torch.sigmoid(model(x).float())\n            return o[0].cpu().numpy()\n        except RuntimeError:\n            torch.cuda.empty_cache()\n            o = torch.sigmoid(model(x).float())\n            return o[0].cpu().numpy()\n    o = torch.sigmoid(model(x).float())\n    return o[0].cpu().numpy()\n\ndef rankpct(x):\n    order = x.argsort(0).argsort(0).astype(np.float64)\n    return order / max(1, x.shape[0] - 1)\n\ndef _make_reader():\n    import pydicom, cv2\n    from pydicom.pixel_data_handlers.util import apply_modality_lut\n\n    def order_and_meta(sdir):\n        fs = glob.glob(sdir + \'/*.dcm\')\n        recs = []\n        ps_list = []\n        for f in fs:\n            try:\n                h = pydicom.dcmread(f, stop_before_pixels=True)\n                iop = getattr(h, \'ImageOrientationPatient\', None)\n                ipp = getattr(h, \'ImagePositionPatient\', None)\n                if iop is not None and ipp is not None and (len(iop) == 6):\n                    r = np.array(iop[:3], float)\n                    c = np.array(iop[3:], float)\n                    n = np.cross(r, c)\n                    pos = float(np.dot(np.array(ipp, float), n))\n                else:\n                    pos = float(getattr(h, \'InstanceNumber\', 0) or 0)\n                ps = getattr(h, \'PixelSpacing\', None)\n                ps = float(ps[0]) if ps is not None else 0.5\n                ps_list.append(ps)\n                recs.append((pos, f, ps))\n            except Exception:\n                recs.append((0.0, f, 0.5))\n        recs.sort(key=lambda x: x[0])\n        med_ps = float(np.median(ps_list)) if ps_list else 0.5\n        return ([(f, ps) for _, f, ps in recs], med_ps)\n\n    def read_px(f):\n        d = pydicom.dcmread(f)\n        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n        if str(getattr(d, \'PhotometricInterpretation\', \'\')) == \'MONOCHROME1\':\n            a = a.max() - a\n        return a\n\n    def mm_crop_resize(a, ps):\n        h, w = a.shape\n        cpx = int(round(CROP_MM / max(ps, 0.001)))\n        cpx = min(cpx, min(h, w))\n        y0 = (h - cpx) // 2\n        x0 = (w - cpx) // 2\n        a = a[y0:y0 + cpx, x0:x0 + cpx]\n        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)\n    return (order_and_meta, read_px, mm_crop_resize)\n\ndef _pick_series_for_slot(rows, plane, fluid, used):\n    cands = [r for r in rows if r[\'Anatomical_Plane\'] == plane and r[\'SeriesInstanceUID\'] not in used]\n    if fluid in (0, 1):\n        pref = [r for r in cands if int(r.get(\'Fluid_Sensitive\', 0) or 0) == fluid]\n        if pref:\n            return pref[0]\n    return cands[0] if cands else None\n\ndef build_study(sid, ser_records, tsdir, reader):\n    order_and_meta, read_px, mm_crop_resize = reader\n    rows = ser_records.get(sid, [])\n    vol = np.zeros((MAXS, IMG, IMG), np.uint8)\n    idx = 0\n    used = set()\n    for plane, fluid, k in SLOTS:\n        r = _pick_series_for_slot(rows, plane, fluid, used)\n        if r is None:\n            idx += k\n            continue\n        used.add(r[\'SeriesInstanceUID\'])\n        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r[\'SeriesInstanceUID\']}")\n        if not files:\n            idx += k\n            continue\n        n = len(files)\n        lo, hi = (int(n * 0.06), int(n * 0.94) - 1)\n        hi = max(hi, lo)\n        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k\n        arrs = []\n        pss = []\n        for p in picks:\n            fp, ps = files[min(p, n - 1)]\n            try:\n                arrs.append(read_px(fp))\n                pss.append(ps)\n            except Exception:\n                arrs.append(None)\n                pss.append(med_ps)\n        valid = [a for a in arrs if a is not None]\n        if valid:\n            allpx = np.concatenate([a.ravel() for a in valid])\n            loq, hiq = np.percentile(allpx, [2.0, 98.0])\n        else:\n            loq, hiq = (0.0, 1.0)\n        for a, ps in zip(arrs, pss):\n            if idx >= MAXS:\n                break\n            if a is None:\n                idx += 1\n                continue\n            aw = np.clip((a - loq) / (hiq - loq + 1e-06), 0, 1)\n            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)\n            vol[idx] = (aw * 255).astype(np.uint8)\n            idx += 1\n        if idx >= MAXS:\n            break\n    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)\n    return (vol, mask)\n\ndef find_test_root():\n    cands = [\'/kaggle/input/competitions/rsna-knee-abnormality-detection\', \'/kaggle/input/rsna-knee-abnormality-detection\']\n    for b in cands:\n        if os.path.exists(b + \'/test.csv\'):\n            return b\n    for d, _, f in os.walk(\'/kaggle/input\'):\n        if \'test.csv\' in f and (os.path.isdir(d + \'/test_series\') or os.path.isdir(d + \'/test_images\')):\n            return d\n    for d, _, f in os.walk(\'/kaggle/input\'):\n        if \'test.csv\' in f:\n            return d\n    raise RuntimeError(\'no test root under /kaggle/input\')\n\ndef find_weight_file(fname):\n    direct = [f\'/kaggle/input/raptor-knee-arms/{fname}\', f\'/kaggle/input/raptor-knee-arms/1/{fname}\', f\'/kaggle/input/raptor-cnn336/{fname}\']\n    for p in direct:\n        if os.path.exists(p):\n            return p\n    for d in sorted(glob.glob(\'/kaggle/input/*/\')):\n        if \'competition\' in d.lower():\n            continue\n        hits = glob.glob(os.path.join(d, \'**\', fname), recursive=True)\n        if hits:\n            return hits[0]\n    raise RuntimeError(f\'{fname} not found under /kaggle/input\')\n\ndef main():\n    import pandas as pd\n    t0 = time.time()\n    dev = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    ngpu = torch.cuda.device_count()\n    print(f\'device {dev} | gpus {ngpu} | torch {torch.__version__}\', flush=True)\n    ROOT = find_test_root()\n    tsdir = ROOT + \'/test_series\'\n    if not os.path.isdir(tsdir):\n        tsdir = ROOT + \'/test_images\'\n    print(\'test root:\', ROOT, \'| series dir:\', tsdir, flush=True)\n    test = pd.read_csv(ROOT + \'/test.csv\')\n    test[\'StudyInstanceUID\'] = test[\'StudyInstanceUID\'].astype(str)\n    test_ids = test[\'StudyInstanceUID\'].tolist()\n    tser = pd.read_csv(ROOT + \'/test_series.csv\')\n    tser[\'StudyInstanceUID\'] = tser[\'StudyInstanceUID\'].astype(str)\n    tser[\'SeriesInstanceUID\'] = tser[\'SeriesInstanceUID\'].astype(str)\n    SER = {k: v.to_dict(\'records\') for k, v in tser.groupby(\'StudyInstanceUID\')}\n    print(f\'test studies {len(test_ids)} | test series {len(tser)}\', flush=True)\n    sub_cols = [\'StudyInstanceUID\'] + LAB\n    ssub = os.path.join(ROOT, \'sample_submission.csv\')\n    if os.path.exists(ssub):\n        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)\n    reader = _make_reader()\n    N = len(test_ids)\n    A = len(ARMS)\n    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]\n    for a, arm in enumerate(ARMS):\n        wp = find_weight_file(arm[\'file\'])\n        model, res = load_model(wp, arm[\'arch\'], arm[\'res\'], dev)\n        print(f"[arm {a}] loaded {arm[\'file\']} | res {res} | {time.time() - t0:.0f}s", flush=True)\n        for i, sid in enumerate(test_ids):\n            try:\n                vol, mask = build_study(sid, SER, tsdir, reader)\n                xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM)\n                arm_probs[a][i] = infer_probs(model, xw, dev)\n                del vol, mask, xw\n            except Exception as e:\n                print(f\'  [arm {a}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})\', flush=True)\n            if (i + 1) % 100 == 0 or i + 1 == N:\n                print(f\'  [arm {a}] {i + 1}/{N} | {time.time() - t0:.0f}s\', flush=True)\n        del model\n        gc.collect()\n        if str(dev).startswith(\'cuda\'):\n            torch.cuda.empty_cache()\n        print(f\'[arm {a}] done + freed | {time.time() - t0:.0f}s\', flush=True)\n    _w = np.array([float(a.get(\'w\', 1.0)) for a in ARMS], dtype=np.float64)\n    _w = _w / _w.sum()\n    print(f"[blend] weighted rank-mean w={dict(zip([a[\'file\'] for a in ARMS], _w.round(4)))}", flush=True)\n    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]), axes=(0, 0))\n    if not np.isfinite(ranks).all():\n        ranks[~np.isfinite(ranks)] = 0.5\n    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)\n    sub.insert(0, \'StudyInstanceUID\', test_ids)\n    sub = sub[sub_cols]\n    assert list(sub.columns) == sub_cols, \'column order drift\'\n    assert sub[\'StudyInstanceUID\'].tolist() == test_ids, \'row identity drift\'\n    assert np.isfinite(sub[LAB].values).all()\n    out = \'/kaggle/working/_raptor.csv\'\n    sub.to_csv(out, index=False)\n    print(\'wrote\', out, \'|\', len(sub), \'rows x\', len(sub.columns), \'cols\', flush=True)\n    print(sub.head().to_string(index=False), flush=True)\n    print(f\'DONE {time.time() - t0:.0f}s\', flush=True)'
WORKER_SRC = '\nimport os, sys, time, glob, math, random, hashlib, json, copy\nimport numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F\nt0 = time.time()\nGTAG = os.environ.get(\'GTAG\', \'?\')\ndef tlog(m): print(f\'[g{GTAG} t={time.time()-t0:.0f}s] {m}\', flush=True)\n\nKE = {\'__name__\': \'_ke\'}\nexec(compile(open(\'/kaggle/working/_ke_src.py\').read(), \'<ke>\', \'exec\'), KE)\nLAB = KE[\'LAB\']; N_T = len(LAB)\nCFG = json.loads(os.environ[\'CFG\'])\nARCH, RES, EPOCHS, SEED, NFOLD = CFG[\'arch\'], int(CFG[\'res\']), int(CFG[\'epochs\']), int(CFG[\'seed\']), int(CFG[\'nfold\'])\nFOLDS = [int(x) for x in os.environ[\'FOLDS\'].split(\',\') if x != \'\']\nLR_BB, LR_HEAD, WD = float(CFG[\'lr_bb\']), 2e-3, 0.02\nK_WIN, BS, ACC, K_EVAL, WARMUP = int(CFG.get(\'kwin\', 12)), int(CFG.get(\'bs\', 4)), int(CFG.get(\'acc\', 2)), 42, int(CFG.get(\'warmup\', 150))\nimport torch_xla.core.xla_model as xm\nDEV = xm.xla_device()\nNW = int(os.environ.get(\'NW\', \'3\'))\nOUTTAG = CFG[\'outtag\']\ntlog(f\'worker mulai | arch={ARCH} res={RES} epochs={EPOCHS} seed={SEED} nfold={NFOLD} folds={FOLDS}\')\n\ndef _bglob(name):\n    return [p for d in (\'/kaggle/input/*/\', \'/kaggle/input/*/*/\', \'/kaggle/input/*/*/*/\', \'/kaggle/input/*/*/*/*/\')\n            for p in glob.glob(d + name)]\n\n# ---------------- data ----------------\nROOT = KE[\'find_test_root\']()\ntrain = pd.read_csv(ROOT + \'/train.csv\'); train[\'StudyInstanceUID\'] = train[\'StudyInstanceUID\'].astype(str)\ntser = pd.read_csv(ROOT + \'/train_series.csv\'); tser[\'StudyInstanceUID\'] = tser[\'StudyInstanceUID\'].astype(str)\nHAS_SER = set(tser[\'StudyInstanceUID\'])\nGOLD = set(train.loc[train[LAB].notna().all(axis=1), \'StudyInstanceUID\'])\nALL_IDS = [s for s in train[\'StudyInstanceUID\'] if s in HAS_SER]\nMOUNTED = {}\nfor d in (glob.glob(\'/kaggle/input/*/volcache\') + glob.glob(\'/kaggle/input/*/*/volcache\') + glob.glob(\'/kaggle/input/*/*/*/volcache\')):\n    if os.path.isdir(d):\n        for f in os.listdir(d):\n            if f.endswith(\'.npz\'): MOUNTED[f[:-4]] = os.path.join(d, f)\ntlog(f\'studi={len(ALL_IDS)} gold={len(GOLD)} cache termount={len(MOUNTED)}\')\nhits = _bglob(CFG[\'label_file\']); assert hits, \'csv label tak ketemu\'\nFL = pd.read_csv(hits[0]); FL[\'StudyInstanceUID\'] = FL[\'StudyInstanceUID\'].astype(str); FL = FL.set_index(\'StudyInstanceUID\')\ntlog(\'label csv: \' + hits[0])\nweak_ids = [s for s in ALL_IDS if s not in GOLD and s in FL.index]\nmiss = [s for s in weak_ids + sorted(GOLD) if s not in MOUNTED]\nassert not miss, f\'cache hilang utk {len(miss)} studi\'\ngold_ids = sorted(GOLD)\nSMOKE = bool(CFG.get(\'smoke\'))\nif SMOKE: gold_ids = gold_ids[:16]\n\nrep = train.set_index(\'StudyInstanceUID\')[\'Report\'] if \'Report\' in train.columns else None\ndef gkey(s):\n    r = None if rep is None else rep.get(s)\n    if r is None or isinstance(r, float) or not str(r).strip(): return \'u:\' + s\n    return \'r:\' + hashlib.md5(\' \'.join(str(r).split()).encode()).hexdigest()\ngroups = {}\nfor s in weak_ids: groups.setdefault(gkey(s), []).append(s)\nglist = sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0]))\nsizes = [0] * NFOLD; fold_of = {}\nfor gk, members in glist:\n    f = int(np.argmin(sizes))\n    for m in members: fold_of[m] = f\n    sizes[f] += len(members)\ntlog(f\'grup={len(groups)} studi_lemah={len(weak_ids)} ukuran_fold={sizes}\')\n\nYcols = [t for t in LAB]\ndef label_arrays(ids):\n    y = FL.loc[ids, Ycols].fillna(0.28).values.astype(np.float32)\n    c = np.stack([FL.loc[ids, f\'{t}__conf\'].fillna(0.15).values.astype(np.float32) for t in LAB], 1)\n    return y, (0.5 + np.clip(c, 0, 1)).astype(np.float32)\n\nMEAN = KE[\'_MEAN\']; STD = KE[\'_STD\']\nif CFG.get(\'norm\') == \'rad\':\n    MEAN = torch.full((3, 1, 1), 0.5); STD = torch.full((3, 1, 1), 0.5)\n\ndef _resize(t, res):\n    if t.shape[-1] != res:\n        t = F.interpolate(t[None], size=(res, res), mode=\'bilinear\', align_corners=False, antialias=True)[0]\n    return t\n\ndef rand_windows(vol, mask, k, res):\n    D = vol.shape[0]\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3: valid = np.arange(min(3, D))\n    lo, hi = int(valid.min()), int(valid.max())\n    cs = list(range(max(1, lo), min(D - 1, hi)))\n    if not cs: cs = [max(1, min(D - 2, (lo + hi) // 2))]\n    cs = list(np.random.choice(cs, size=min(k, len(cs)), replace=False))\n    while len(cs) < k: cs.append(cs[np.random.randint(len(cs))])\n    H = vol.shape[-1]\n    wins = torch.empty((k, 3, res, res), dtype=torch.float32)\n    for j, c in enumerate(cs):\n        t = torch.from_numpy(np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0)\n        fr = random.uniform(0.88, 1.0); s = int(round(H * fr)); y0 = random.randint(0, H - s); x0 = random.randint(0, H - s)\n        t = t[:, y0:y0 + s, x0:x0 + s]\n        t = _resize(t, res)\n        m_ = t.mean(); t = ((t - m_) * random.uniform(0.85, 1.15) + m_) * random.uniform(0.85, 1.15)\n        wins[j] = (t.clamp(0, 1) - MEAN) / STD\n    return wins\n\ndef eval_windows_aa(vol, mask, k, res):\n    D = vol.shape[0]\n    cs = KE[\'_eval_centers\'](mask, D, k)\n    wins = torch.empty((len(cs), 3, res, res), dtype=torch.float32)\n    for j, c in enumerate(cs):\n        c = max(1, min(c, D - 2))\n        t = torch.from_numpy(np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0)\n        wins[j] = (_resize(t, res) - MEAN) / STD\n    return wins\n\nclass TrainDS(torch.utils.data.Dataset):\n    def __init__(self, ids, Y, W): self.ids, self.Y, self.W = ids, Y, W\n    def __len__(self): return len(self.ids)\n    def __getitem__(self, i):\n        z = np.load(MOUNTED[self.ids[i]]); vol, mask = z[\'vol\'], z[\'mask\']\n        return rand_windows(vol, mask, K_WIN, RES), torch.from_numpy(self.Y[i]), torch.from_numpy(self.W[i])\n\nclass EvalDS(torch.utils.data.Dataset):\n    def __init__(self, ids): self.ids = ids\n    def __len__(self): return len(self.ids)\n    def __getitem__(self, i):\n        z = np.load(MOUNTED[self.ids[i]]); vol, mask = z[\'vol\'], z[\'mask\']\n        x_ = eval_windows_aa(vol, mask, K_EVAL, RES)\n        if x_.shape[0] < K_EVAL: x_ = x_[[i % x_.shape[0] for i in range(K_EVAL)]]\n        return x_\n\ndef worker_init(_):\n    s = (torch.initial_seed() + 12345) % (2 ** 31)\n    np.random.seed(s); random.seed(s)\n\n# ---------------- model ----------------\ndef find_weights(rel, match=\'\'):\n    hits = set()\n    for base in (\'/kaggle/input/\', \'/kaggle/input/*/\', \'/kaggle/input/*/*/\', \'/kaggle/input/*/*/*/\', \'/kaggle/input/*/*/*/*/\'):\n        for p in glob.glob(base + rel):\n            if match in p.lower(): hits.add(os.path.realpath(p))\n    return sorted(hits)\n\ndef make_model():\n    bb = KE[\'build_backbone\'](ARCH, pretrained=False)\n    wf = find_weights(CFG[\'rel\'], CFG[\'match\'])\n    assert len(wf) == 1, f"bobot pra-latih harus tepat 1: {wf}"\n    if CFG.get(\'fmt\') == \'rad\':\n        raw = torch.load(wf[0], map_location=\'cpu\', weights_only=True); sd = {}\n        for k, v in raw.items():\n            k2 = k[len(\'backbone.\'):] if k.startswith(\'backbone.\') else k\n            head, _, rest = k2.partition(\'.\')\n            if head == \'0\': sd[\'conv1.\' + rest] = v\n            elif head == \'1\': sd[\'bn1.\' + rest] = v\n            elif head in (\'4\', \'5\', \'6\', \'7\'): sd[f\'layer{int(head) - 3}.\' + rest] = v\n            else: raise RuntimeError(f\'kunci RadImageNet tak dikenal: {k}\')\n    else:\n        from safetensors.torch import load_file\n        sd = load_file(wf[0])\n    r = bb.load_state_dict(sd, strict=False)\n    tlog(f\'bobot {wf[0][-70:]}: missing={list(r.missing_keys)[:5]} unexpected={list(r.unexpected_keys)[:5]}\')\n    assert len(r.missing_keys) == 0, \'bobot tak lengkap\'\n    return KE[\'RaptorClassifier\'](bb, F_dim=bb.num_features)\n\nclass EMA:\n    def __init__(self, model, decay=0.998):\n        self.decay = decay; self.n = 0\n        self.sd = {k: v.detach().clone().float() if v.is_floating_point() else v.detach().clone() for k, v in model.state_dict().items()}\n    @torch.no_grad()\n    def update(self, model):\n        self.n += 1; d = self.decay\n        for k, v in model.state_dict().items():\n            if v.is_floating_point(): self.sd[k].mul_(d).add_(v.detach().float(), alpha=1 - d)\n            else: self.sd[k].copy_(v)\n\ndef auc1(yy, ss):\n    pos, neg = ss[yy == 1], ss[yy == 0]\n    if len(pos) == 0 or len(neg) == 0: return np.nan\n    r = pd.Series(np.concatenate([pos, neg])).rank().values\n    return float((r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))\nYg = train.set_index(\'StudyInstanceUID\').loc[gold_ids, LAB].values.astype(float)\ndef macro(P, Y): return float(np.nanmean([auc1(Y[:, j], P[:, j]) for j in range(N_T)]))\n\n@torch.no_grad()\ndef predict(model_, ids):\n    model_.eval(); out = []\n    dl = torch.utils.data.DataLoader(EvalDS(ids), batch_size=1, shuffle=False, num_workers=NW, worker_init_fn=worker_init)\n    for xw in dl:\n        try:\n            with torch.autocast(\'xla\', dtype=torch.bfloat16):\n                p = torch.sigmoid(model_(xw.to(DEV)).float())\n            out.append(p[0].cpu().numpy())\n        except Exception as e:\n            tlog(f\'eval gagal: {type(e).__name__}: {str(e)[:100]}\'); out.append(np.full(N_T, 0.5, np.float32))\n    return np.stack(out)\n\n# ---------------- latih per fold ----------------\nfor FOLD in FOLDS:\n    seed = SEED * 100 + FOLD\n    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)\n    train_ids = [s for s in weak_ids if fold_of[s] != FOLD]\n    held_ids = [s for s in weak_ids if fold_of[s] == FOLD]\n    assert not ({gkey(s) for s in train_ids} & {gkey(s) for s in held_ids}), \'BOCOR grup laporan\'\n    assert not (set(train_ids) & GOLD), \'gold bocor ke latih\'\n    if SMOKE: train_ids, held_ids = train_ids[:48], held_ids[:16]\n    Y, W = label_arrays(train_ids)\n    tlog(f\'=== FOLD {FOLD}/{NFOLD}: latih={len(train_ids)} held-out={len(held_ids)} gold={len(gold_ids)}\')\n    model = make_model().to(DEV)\n    if CFG.get(\'bias_init\'):\n        pr = np.clip((W * Y).sum(0) / W.sum(0), 0.02, 0.98)\n        with torch.no_grad(): model.clsb.copy_(torch.logit(torch.tensor(pr, dtype=torch.float32)).to(DEV))\n        tlog(\'bias kepala diinisialisasi ke logit prior: \' + \' \'.join(f\'{x:.2f}\' for x in pr))\n    head_params = list(model.norm.parameters()) + list(model.att.parameters()) + [model.clsW, model.clsb]\n    hp = {id(p) for p in head_params}\n    bb_decay = [p for p in model.backbone.parameters() if p.ndim > 1]\n    bb_nodecay = [p for p in model.backbone.parameters() if p.ndim <= 1]\n    opt = torch.optim.AdamW([{\'params\': bb_decay, \'lr\': LR_BB, \'weight_decay\': WD},\n                             {\'params\': bb_nodecay, \'lr\': LR_BB, \'weight_decay\': 0.0},\n                             {\'params\': head_params, \'lr\': LR_HEAD, \'weight_decay\': WD}])\n    base_lrs = [g[\'lr\'] for g in opt.param_groups]\n    steps_per_epoch = len(train_ids) // (BS * ACC)\n    total_steps = steps_per_epoch * EPOCHS\n    def set_lr(s):\n        sc = min(1.0, (s + 1) / WARMUP) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total_steps)))\n        sc = max(1, round(sc * 20)) / 20\n        for g, b in zip(opt.param_groups, base_lrs): g[\'lr\'] = b * sc\n    ema = EMA(model, float(CFG.get(\'ema\', 0.998))); ema_model = copy.deepcopy(model)\n    dl = torch.utils.data.DataLoader(TrainDS(train_ids, Y, W), batch_size=BS, shuffle=True, drop_last=True,\n                                     num_workers=NW, worker_init_fn=worker_init, prefetch_factor=3, persistent_workers=True)\n    tlog(f\'steps={total_steps} ({steps_per_epoch}/epoch) lr_bb={LR_BB} lr_head={LR_HEAD}\')\n    step = 0\n    for ep in range(EPOCHS):\n        model.train(); torch.backends.cudnn.benchmark = True\n        ep_loss, nb = torch.zeros((), device=DEV), 0; t_ep = time.time()\n        opt.zero_grad(set_to_none=True)\n        for bi, (xb, yb, wb) in enumerate(dl):\n            xb, yb, wb = xb.to(DEV, non_blocking=True), yb.to(DEV), wb.to(DEV)\n            with torch.autocast(\'xla\', dtype=torch.bfloat16):\n                logits = model(xb)\n            loss = (F.binary_cross_entropy_with_logits(logits.float(), yb, reduction=\'none\') * wb).mean()\n            (loss / ACC).backward()\n            ep_loss = ep_loss + loss.detach(); nb += 1\n            if (bi + 1) % ACC == 0:\n                set_lr(step)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n                opt.step(); opt.zero_grad(set_to_none=True); ema.update(model); step += 1\n            xm.mark_step()\n            if (bi + 1) % 200 == 0:\n                tlog(f\'f{FOLD} ep{ep+1} {(bi+1)*BS}/{len(train_ids)} loss~{float(ep_loss)/nb:.4f} lr_bb={opt.param_groups[0]["lr"]:.2e} {(time.time()-t_ep):.0f}s\')\n        torch.backends.cudnn.benchmark = False\n        ema_model.load_state_dict({k: v.to(ema_model.state_dict()[k].dtype) for k, v in ema.sd.items()})\n        Pr = predict(model, gold_ids); Pe = predict(ema_model, gold_ids)\n        tlog(f\'=== f{FOLD} EPOCH {ep+1}/{EPOCHS} loss={float(ep_loss)/max(1,nb):.4f} ({time.time()-t_ep:.0f}s) | gold-58 diag: raw={macro(Pr,Yg):.4f} ema={macro(Pe,Yg):.4f}\')\n    del dl\n    # simpan EMA + OOF\n    torch.save({\'arch\': ARCH, \'res\': RES, \'norm\': CFG.get(\'norm\', \'imagenet\'), \'model\': {k: v.detach().cpu() for k, v in ema_model.state_dict().items()}},\n               f\'/kaggle/working/{OUTTAG}_f{FOLD}.pt\')\n    Ph = predict(ema_model, held_ids); Pg = predict(ema_model, gold_ids)\n    oof = pd.DataFrame(np.vstack([Ph, Pg]), columns=LAB)\n    oof.insert(0, \'StudyInstanceUID\', held_ids + gold_ids)\n    oof[\'is_gold\'] = [0] * len(held_ids) + [1] * len(gold_ids); oof[\'fold\'] = [FOLD] * len(held_ids) + [-1] * len(gold_ids)\n    oof.to_csv(f\'/kaggle/working/oof_{OUTTAG}_f{FOLD}.csv\', index=False)\n    Th = FL.loc[held_ids, LAB].fillna(0.28).values.astype(float)\n    cs = [np.corrcoef(Ph[:, j], Th[:, j])[0, 1] for j in range(N_T)]\n    tlog(f\'*** FOLD {FOLD} SELESAI | gold-58 (EMA) = {macro(Pg, Yg):.4f} | korelasi(OOF, label fusi) rata2 = {np.nanmean(cs):.4f}\')\n    del model, ema_model, opt, ema\ntlog(\'WORKER DONE\')\n'
CFG = {'arch': 'resnet50', 'res': 224, 'epochs': 12, 'seed': 1, 'nfold': 6, 'lr_bb': 0.0002, 'rel': 'ResNet50.pt', 'match': '', 'fmt': 'rad', 'norm': 'rad', 'label_file': 'fused_labels_r1.csv', 'outtag': 'cheap_radresnet50_ep12_s1', 'smoke': False, 'bs': 2, 'acc': 2, 'warmup': 200, 'kwin': 12, 'bias_init': 1, 'ema': 0.998, 'tpu': 1, 'only_folds': [3, 4, 5], 'probe_fold': 0}
CFG_B = None


In [ ]:

import os, sys, time, glob, json, subprocess
import numpy as np, pandas as pd
t0 = time.time()
def tlog(m): print(f'[main t={time.time()-t0:.0f}s] {m}', flush=True)
os.environ.setdefault('HF_HUB_OFFLINE', '1')
open('/kaggle/working/_ke_src.py', 'w').write(KE_SRC)
open('/kaggle/working/worker.py', 'w').write(WORKER_SRC)
import torch
tlog(f'GPU terlihat: {torch.cuda.device_count()} | {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}')
NG = max(1, torch.cuda.device_count())
folds = list(range(CFG['nfold'])) if not CFG.get('smoke') else list(range(min(2, CFG['nfold'])))
if CFG.get('only_folds'): folds = list(CFG['only_folds'])
if CFG.get('tpu'): os.environ.setdefault('PJRT_DEVICE', 'TPU'); os.environ.setdefault('NW', '8')
assign = {g: [f for f in folds if f % NG == g] for g in range(NG)}
CFGS = {g: CFG for g in range(NG)}
if CFG_B is not None and NG >= 2:      # mode probe: dua resep pada fold yg SAMA
    assign = {0: [CFG['probe_fold']], 1: [CFG['probe_fold']]}; CFGS = {0: CFG, 1: CFG_B}
tlog(f'pembagian fold per GPU: {assign} | probe={CFG_B is not None}')
procs = {}
for g, fl in assign.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), FOLDS=','.join(map(str, fl)), GTAG=str(g), CFG=json.dumps(CFGS[g]))
    fh = open(f'/kaggle/working/worker{g}.log', 'w')
    procs[g] = (subprocess.Popen([sys.executable, '-u', '/kaggle/working/worker.py'], env=env, stdout=fh, stderr=subprocess.STDOUT), fh)
pos = {g: 0 for g in procs}
def pump():
    for g in procs:
        with open(f'/kaggle/working/worker{g}.log') as fh:
            fh.seek(pos[g]); new = fh.read(); pos[g] = fh.tell()
        for line in new.splitlines(): print(line, flush=True)
while any(p.poll() is None for p, _ in procs.values()):
    time.sleep(10); pump()
pump()
for g, (p, fh) in procs.items(): tlog(f'pekerja g{g} selesai rc={p.returncode}')

# ---- gabung: ensemble gold-58 lintas fold ----
LAB = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
files = sorted(glob.glob(f"/kaggle/working/oof_{CFG['outtag']}_f*.csv"))
if CFG_B is not None:
    tlog('MODE PROBE: ringkasan per resep dilewati (lihat baris "FOLD ... SELESAI" tiap GPU)'); files = []
tlog(f'berkas OOF: {len(files)}')
if files:
    ROOT = [p for p in ('/kaggle/input/competitions/rsna-knee-abnormality-detection', '/kaggle/input/rsna-knee-abnormality-detection') if os.path.exists(p + '/train.csv')][0]
    train = pd.read_csv(ROOT + '/train.csv'); train['StudyInstanceUID'] = train['StudyInstanceUID'].astype(str)
    def auc1(yy, ss):
        pos_, neg = ss[yy == 1], ss[yy == 0]
        r = pd.Series(np.concatenate([pos_, neg])).rank().values
        return float((r[:len(pos_)].sum() - len(pos_) * (len(pos_) + 1) / 2) / (len(pos_) * len(neg)))
    G = []; weak = []
    for f in files:
        o = pd.read_csv(f, dtype={'StudyInstanceUID': str})
        G.append(o[o.is_gold == 1].set_index('StudyInstanceUID')); weak.append(o[o.is_gold == 0])
    ids = list(G[0].index); Y = train.set_index('StudyInstanceUID').loc[ids, LAB].values.astype(float)
    P = np.mean([g.loc[ids, LAB].values for g in G], 0)
    R = np.mean([np.column_stack([pd.Series(g.loc[ids, LAB].values[:, j]).rank(pct=True).values for j in range(12)]) for g in G], 0)
    for i, g in enumerate(G): tlog(f'  fold-model {i}: gold-58 = {np.mean([auc1(Y[:, j], g.loc[ids, LAB].values[:, j]) for j in range(12)]):.4f}')
    a = [auc1(Y[:, j], P[:, j]) for j in range(12)]
    tlog(f'ENSEMBLE {len(G)} fold | gold-58 prob-mean = {np.mean(a):.4f} | rank-mean = {np.mean([auc1(Y[:, j], R[:, j]) for j in range(12)]):.4f}   [acuan: c5 stok 0.9213, 4-murid-CoAtNet 0.9286, v3 0.9234]')
    for l, v in zip(LAB, a): tlog(f'    {l:18s} {v:.4f}')
    pd.concat(weak).to_csv(f"/kaggle/working/oof_weak_{CFG['outtag']}_all.csv", index=False)
tlog('SELESAI')
